In [1]:
# les imports
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

In [2]:
# les réglages du modèle et de l'entraînement
batch_size = 16
block_size = 64
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1

max_iters = 3000
eval_interval = 250
learning_rate = 1e-3
eval_iters = 50

In [3]:
# on lit le fichier d'excuses
with open('data/excuses.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

In [4]:
# test : longueur du texte
# print("longueur du texte en caractères :", len(text))

In [5]:
# test : le début du texte
# print(text[:500])

In [6]:
# test : quelques infos sur les excuses
# lignes = text.splitlines()
# print("nombres d'excuses :", len(lignes))
# print("excuse la plus courte :", min(lignes, key=len))
# print("excuse la plus longue :", max(lignes, key=len))

In [7]:
# le découpeur de gpt-2 (tiktoken)
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
# print("taille du vocabulaire :", vocab_size)

In [8]:
# test : comment tiktoken découpe une phrase
# exemple = "mon chat a mangé mes devoirs"
# decoupage = enc.encode(exemple)
# print(decoupage)
# print(len(exemple), "caractères ->", len(decoupage), "mots")
# print([enc.decode([t]) for t in decoupage])
# print(enc.decode(decoupage))


In [9]:
# tout le texte devient des tokens
data = torch.tensor(enc.encode(text), dtype=torch.long)
# print(data.shape, data.dtype)
# print(data[:12])

In [10]:
# on sépare train (90%) et val (10%)
n = int(0.9 * len(data)) 
train_data = data[:n]
val_data = data[n:]
# print(len(train_data), len(val_data))

In [11]:
# test : longueur des excuses en tokens
# longueurs = [len(enc.encode(l)) for l in lignes]
# print("l'excuse la plus longue est :", max(longueurs), "tokens")
# print("l'excuse la plus courte est :", min(longueurs), "tokens")

In [12]:
# fonction qui prend des petits morceaux au hasard
torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+ block_size] for i in ix])
    y = torch.stack([data[i+1:i + block_size +1] for i in ix])
    return x,y

In [13]:
# test : regarder un lot
# xb, yb = get_batch('train')
# print(xb.shape)
# print(xb)
# print(yb)

In [14]:
# test : ce que le modèle doit deviner
# for t in range(block_size) :
#     contexte = xb[0, :t+1]
#     cible = yb[0, t]
#     print(repr(enc.decode(contexte.tolist())), "->", repr(enc.decode([cible.item()])))

In [15]:
# ancien test : le modèle bigramme
# from transformer_model import BigramLanguageModel

In [16]:
# ancien test : le bigramme avant entraînement
# m = BigramLanguageModel(vocab_size, n_embd)
# print("nombre de poids :", sum(p.numel() for p in m.parameters()))
# logits, loss = m(xb, yb)
# print(logits.shape)
# print(loss)

In [17]:
# ancien test : à quoi ressemble une fiche
# print(m.token_embedding_table.weight.shape)
# print(m.token_embedding_table.weight[325][:6])

In [18]:
# ancien test : le bigramme écrit n'importe quoi
# idx = torch.tensor([[198]]) 
# print(enc.decode(m.generate(idx, max_new_tokens=25)[0].tolist()))

In [19]:
# ancien test : l'optimizer du bigramme
# optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [20]:
# ancien test : entraînement du bigramme
# for steps in range(300):
#     xb, yb = get_batch('train')
#     logits, loss = m(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()
#     if steps % 50 == 0:
#         print(steps, loss.item())

In [21]:
# ancien test : le bigramme après entraînement
# idx = torch.tensor([[198]])
# print(enc.decode(m.generate(idx, max_new_tokens=30)[0].tolist()))

In [22]:
# test : l'attention à la main
# torch.manual_seed(1337)
# B, T, C = 4, 8, 32
# x = torch.randn(B, T, C)

# head_size = 16
# key = nn.Linear(C, head_size, bias=False)
# query = nn.Linear(C, head_size, bias=False)
# value = nn.Linear(C, head_size, bias=False)

# k = key(x)
# q = query(x)
# wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [23]:
# test : le masque (on cache le futur)
# tril = torch.tril(torch.ones(T, T))
# wei = wei.masked_fill(tril == 0, float('-inf'))
# wei = F.softmax(wei, dim=-1)
# print(wei[0].round(decimals=2))

# out = wei @ value(x)
# print(out.shape)

In [24]:
# on importe le modèle
from transformer_model import GPTLanguageModel

In [25]:
# on crée le modèle
m = GPTLanguageModel(vocab_size, n_embd, block_size, n_head, n_layer, dropout)
print("nombre de poids :", sum(p.numel() for p in m.parameters()))

# for nom, poids in m.state_dict().items():
#     if 'heads.0.' in nom or 'proj' in nom or 'ffwd' in nom:
#         print(nom, tuple(poids.shape))

nombre de poids : 13716049


In [26]:
# ancien test : 300 pas pour voir
# optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# for steps in range(300):
#     xb, yb = get_batch('train')
#     logits, loss = m(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()
#     if steps % 50 == 0:
#         print(steps, loss.item())

# idx = torch.tensor([[198]])
# print(enc.decode(m.generate(idx, max_new_tokens=30)[0].tolist()))

In [27]:
# test : combien de temps prend un pas
# import time

# optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# t0 = time.time()
# for steps in range(20):
#     xb, yb = get_batch('train')
#     logits, loss = m(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()
# duree = (time.time() - t0) / 20

# print(f"{duree:.2f} secondes par pas")
# for n in [500, 1000, 2000, 3000, 6000]:
#     print(n, "pas ->", round(n * duree / 60), "minutes")

In [28]:
# test : combien de pas pour relire tout le texte
# tokens_par_pas = batch_size * block_size
# print(tokens_par_pas, "tokens par pas")
# print(round(len(train_data) / tokens_par_pas), "pas pour voir autant de tokens que tout le train")

In [30]:
# fonction pour mesurer la loss sur train et val
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [31]:
# entraînement
torch.manual_seed(1337)
m = GPTLanguageModel(vocab_size, n_embd, block_size, n_head, n_layer, dropout)
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

historique = []
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        historique.append((iter, losses['train'].item(), losses['val'].item()))
        print(f"pas {iter}: train {losses['train']:.4f}, val {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

pas 0: train 10.9923, val 10.9952
pas 250: train 1.2492, val 1.2754
pas 500: train 0.6929, val 0.7206
pas 750: train 0.5809, val 0.6312
pas 1000: train 0.5349, val 0.5944
pas 1250: train 0.5246, val 0.5663
pas 1500: train 0.5433, val 0.5849
pas 1750: train 0.5254, val 0.5738
pas 2000: train 0.5044, val 0.5869
pas 2250: train 0.5101, val 0.5479
pas 2500: train 0.4896, val 0.5719
pas 2750: train 0.5036, val 0.5409
pas 2999: train 0.4816, val 0.5480


In [32]:
# 5 excuses pour voir
m.eval()
idx = torch.tensor([[198]])
for _ in range(5):
    print(enc.decode(m.generate(idx, max_new_tokens=60)[0].tolist()))


Je ne peux pas t'aider à déménager parce que ma grand-mère a décidé d'apprendre la trompette à six heures du matin.
Je n'ai pas fait mes devoirs parce qu

Excuse-moi pour ce matin, mais mon oreiller a remplacé ma patience par une pastèque.
Je n'ai pas appelé ma mère car une poule ambitieuse a mangé ma carte de bus.
Je

Je n'ai pas pu rendre mon devoir parce que j'étais occupé à garder le moral de ma plante (c'est la troisième fois ce mois-ci).
Ne m'en veux pas, c'est mon

Si je suis en retard, c'est qu'un pont s'est faisait sieste, ce qui n'arrive qu'une fois par siècle.
Je n'ai pas fait les courses parce qu'un clown m'a en

Je ne suis pas venu au sport car un hérisson a fait une crise existentielle, comme chaque mardi.
Il pleuvait des grenouilles, alors j'ai préféré rester sous la couette (


In [33]:
# on sauvegarde le modèle
torch.save({
    'model_state': m.state_dict(),
    'config': dict(vocab_size=vocab_size, n_embd=n_embd, block_size=block_size,
                   n_head=n_head, n_layer=n_layer, dropout=dropout),
    'historique': historique,
}, 'gpt_excuses.pt')

In [34]:
# le générateur d'excuses
@torch.no_grad()
def excuse(debut="", temperature=0.8, top_k=50, max_new_tokens=80):
    m.eval()
    idx = torch.tensor([[198] + enc.encode(debut)])
    for _ in range(max_new_tokens):
        logits, _ = m(idx[:, -block_size:])
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        if idx_next.item() == 198:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return enc.decode(idx[0, 1:].tolist())

print(excuse())
print(excuse("Je suis en retard"))
print(excuse("Je n'ai pas fait mes devoirs parce que"))
print(excuse("C'est la faute de mon chat", temperature=1.2))

Je n'ai pas pu faire le ménage parce que j'étais occupé à trier mes chaussettes par humeur, encore une fois.
Je suis en retard parce qu'une parade de canards traversait la ville. Je n'invente rien.
Je n'ai pas fait mes devoirs parce que ma grand-mère a recouvert mon sandwich de confiture. Je suis aussi surpris que toi.
C'est la faute de mon chat : il a ouvert une boulangerie dans ma chambre (c'est la troisième fois ce mois-ci).
